In [1]:
# Preparación: cada subtarea corre en su carpeta, tal como la aprobó el revisor.
import os
from pathlib import Path
from IPython.display import Image, display

RAIZ = Path.cwd()

def mostrar_figuras(carpeta):
    for png in sorted(Path(carpeta).rglob("*.png")):
        display(Image(filename=str(png)))

## RAG mínimo en ~60 líneas — MMIA 6013 · Semana 2 · Miércoles

Pipeline completo: chunking → índice → retrieval → prompt con contexto → LLM. Sin API funciona en «modo inspección»: muestra el prompt armado y una respuesta pre-capturada, para poder depurar el retrieval igual (enunciado, S0). Actividad de clase de la Semana 2 (apartado 2.a del Taller 03 v2), sobre el corpus de políticas internas de NimbusSoft: `politica_vacaciones.md`, `politica_remoto.md` y `gastos.md`.

**Estado de ejecución (declaración obligatoria):**

- **Parte 1 — Chunking: EJECUTADA Y MEDIDA.** Todas las cifras de esta parte provienen de la ejecución real.
- **Parte 2 — Índice (embeddings + kNN): NO EJECUTADA — presupuesto agotado.** Sin cifras medidas.
- **Parte 3 — Generación con contexto: NO EJECUTADA — presupuesto agotado.** Sin cifras medidas.

**Lo que faltó (declaración explícita):**

1. T2 — Parte 2 (índice, techo del modelo y `recuperar()`, Ejercicio 2.1): **presupuesto agotado**.
2. T3 — Parte 3 (generación con contexto y las tres pruebas de fuego, Ejercicio 3.1): **presupuesto agotado**.

Por tanto, este notebook **no** está ejecutado de principio a fin como pide la entrega: la Parte 1 está completa y medida; las Partes 2 y 3 quedan pendientes de ejecución. Donde el enunciado espera una cifra medida y no la hay, se declara «sin medición» en lugar de inventarla.

## Parte 1 — Chunking

In [2]:
# Subtarea T1: código aprobado (se ejecuta en una copia: trabajo_nb/T1/)
os.chdir(RAIZ / 'trabajo_nb' / 'T1')

# -*- coding: utf-8 -*-
"""
T1 · Parte 1 — Chunking fijo con solapamiento (Ejercicio 1.1)
MMIA 6013 · Semana 2 (Miércoles) · RAG mínimo en ~60 líneas

Ejercicio 1.1:
  · Implementar chunk_fijo(texto, tamano=300, overlap=80) sobre caracteres
    (devuelve una lista de strings).
  · Construir el corpus fragmentado FUERA de la solución del ejercicio
    (listas `chunks` y `origen` recorriendo DOCS, tal como da el profesor)
    e imprimir cada chunk como '[i] (origen) primeros 70 caracteres...'.
  · Reportar si algún chunk corta una idea (p. ej. la de días transferibles
    "hasta un máximo de 5 días") y repetir la impresión con overlap=0 para
    comparar los cortes.

Cifras guardadas en resultados.json: num_chunks, chunks_por_documento,
idea_cortada_con_overlap_80, num_chunks_con_overlap_0 (+ complementarias).
Esta subtarea NO produce figuras PNG.
"""

# ----------------------------------------------------------------------
# S0 · Imports y corpus (código dado en el enunciado)
# ----------------------------------------------------------------------
import os
import json
import numpy as np  # dado en S0; esta parte aún no lo usa

# Corpus: documentación interna de una empresa ficticia
DOCS = {
    "politica_vacaciones.md": """Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo acumulan 1.5 días de vacaciones por mes trabajado,
hasta un máximo de 18 días por año. Las vacaciones deben solicitarse con al menos
15 días de anticipación a través del portal interno. Los días no utilizados pueden
transferirse al año siguiente hasta un máximo de 5 días. Durante el primer año,
los días solo pueden tomarse después de superar el período de prueba de 3
meses.""",
    "politica_remoto.md": """Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permitido hasta 3 días por semana para todos los roles
excepto soporte de infraestructura on-site. Los días remotos se coordinan con el
líder de equipo. Para trabajar desde el exterior del país se requiere aprobación
de Recursos Humanos con 30 días de anticipación y un máximo de 60 días por año.""",
    "gastos.md": """Política de reembolso de gastos de NimbusSoft.
Los gastos de viaje se reembolsan presentando factura dentro de los 30 días
posteriores al gasto. El límite diario de alimentación en viajes es de 45 USD.
Los pasajes aéreos deben comprarse en clase económica salvo vuelos de más de
8 horas, donde se permite económica premium con aprobación del gerente de área.""",
}

# ----------------------------------------------------------------------
# Ejercicio 1.1 — chunking fijo con solapamiento (SOLUCIÓN)
# ----------------------------------------------------------------------
TAMANO = 300   # tamaño de chunk en caracteres (notebook de hoy: 300/80)
OVERLAP = 80   # solapamiento en caracteres


def _chunk_spans(texto, tamano, overlap):
    """Spans [inicio, fin) de cada chunk sobre el texto original.

    Única fuente de verdad del corte: la ventana avanza
    `tamano - overlap` caracteres por paso, de modo que cada chunk
    repite los últimos `overlap` caracteres del chunk anterior.
    """
    if not (0 <= overlap < tamano):
        raise ValueError("Se requiere 0 <= overlap < tamano")
    paso = tamano - overlap
    spans = []
    inicio = 0
    while inicio < len(texto):
        spans.append((inicio, min(inicio + tamano, len(texto))))
        inicio += paso
    return spans


def chunk_fijo(texto, tamano=300, overlap=80):
    """Ejercicio 1.1: chunking fijo sobre caracteres con solapamiento.

    Devuelve una lista de strings: cada chunk tiene a lo sumo `tamano`
    caracteres y comparte `overlap` caracteres con el chunk anterior, de
    modo que una idea que cruza una frontera aparece completa en al menos
    un chunk (a cambio de inflar el índice: C/(C-V) = 300/220 -> +36.4 %).
    """
    return [texto[a:b] for a, b in _chunk_spans(texto, tamano, overlap)]


# ----------------------------------------------------------------------
# Herramientas de análisis: ¿algún chunk corta una idea?
# ----------------------------------------------------------------------
# La "idea" de los días transferibles es la oración que empieza en
# "Los días no utilizados pueden" y termina en la cifra clave
# "hasta un máximo de 5 días." (la frase que cita el enunciado).
IDEA_DOC = "politica_vacaciones.md"
IDEA_INI = "Los días no utilizados pueden"
IDEA_FIN = "hasta un máximo de 5 días."
FRASE_CLAVE = "hasta un máximo de 5 días"


def _offsets_globales(overlap):
    """Índice global del primer chunk de cada documento en `chunks`."""
    offs, acc = {}, 0
    for nombre, texto in DOCS.items():
        offs[nombre] = acc
        acc += len(chunk_fijo(texto, TAMANO, overlap))
    return offs


def _clasificar_span(nombre_doc, ini, fin, overlap):
    """Clasifica cada chunk del documento frente al span [ini, fin):
      'completa' -> el chunk contiene el span entero,
      'corta'    -> lo interseca pero no lo contiene entero,
      'fuera'    -> no lo toca.
    Los índices de chunk son GLOBALES (los de las listas chunks/origen).
    """
    texto = DOCS[nombre_doc]
    base = _offsets_globales(overlap)[nombre_doc]
    filas = []
    for j, (a, b) in enumerate(_chunk_spans(texto, TAMANO, overlap)):
        interseca = (a < fin) and (ini < b)
        completa = (a <= ini) and (fin <= b)
        estado = "completa" if completa else ("corta" if interseca else "fuera")
        filas.append({"chunk": base + j, "span": (a, b), "estado": estado})
    return filas


def _reporte_idea(overlap):
    """Imprime y devuelve el análisis de corte de la idea para un overlap."""
    texto = DOCS[IDEA_DOC]
    ini = texto.find(IDEA_INI)
    fin = texto.find(IDEA_FIN) + len(IDEA_FIN)
    assert ini != -1 and fin > ini, "No se encontró la idea en el documento"
    idea = texto[ini:fin].replace("\n", " ")
    filas = _clasificar_span(IDEA_DOC, ini, fin, overlap)

    print("")
    print("-" * 72)
    print(f"¿Algún chunk corta la idea? (overlap={overlap})")
    print("-" * 72)
    print(f"Idea buscada en '{IDEA_DOC}', caracteres {ini} a {fin - 1}:")
    print(f'  "{idea}"')
    cortan, completan = [], []
    for f in filas:
        i, (a, b), est = f["chunk"], f["span"], f["estado"]
        trozo = texto[a:b]
        if est == "completa":
            completan.append(i)
            print(f"  [{i}] contiene la idea COMPLETA (chars {a}-{b - 1})")
        elif est == "corta":
            cortan.append(i)
            if a <= ini:
                cola = trozo[-40:].replace("\n", " ")
                print(f'  [{i}] CORTA la idea: termina en "...{cola}"')
            else:
                cabeza = trozo[:40].replace("\n", " ")
                print(f'  [{i}] CORTA la idea: empieza en "{cabeza}..."')
        else:
            print(f"  [{i}] no toca la idea")
    return {"ini": int(ini), "fin": int(fin), "idea": idea,
            "cortan": cortan, "completan": completan}


# ----------------------------------------------------------------------
# Corpus fragmentado — FUERA del ejercicio: las Partes 2 y 3 lo consumen
# (código tal como lo da el profesor)
# ----------------------------------------------------------------------
chunks, origen = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto):
        chunks.append(c)
        origen.append(nombre)

# Comprobación de consistencia: los spans reproducen el corpus exacto
for nombre, texto in DOCS.items():
    reconstruido = [texto[a:b] for a, b in _chunk_spans(texto, TAMANO, OVERLAP)]
    real = [c for c, o in zip(chunks, origen) if o == nombre]
    assert reconstruido == real, "Inconsistencia entre spans y corpus"

print("=" * 72)
print("CHUNKS DEL CORPUS — chunk_fijo(texto, tamano=300, overlap=80)")
print("=" * 72)
for i, c in enumerate(chunks):
    print(f"[{i}] ({origen[i]}) {c[:70]}...")

# ----------------------------------------------------------------------
# Reporte: ¿algún chunk corta una idea? (overlap=80)
# ----------------------------------------------------------------------
rep80 = _reporte_idea(OVERLAP)
idea_cortada_80 = len(rep80["cortan"]) > 0

# ----------------------------------------------------------------------
# Comparación: repetir la impresión con overlap=0
# ----------------------------------------------------------------------
chunks0, origen0 = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto, overlap=0):
        chunks0.append(c)
        origen0.append(nombre)

print("")
print("=" * 72)
print("CHUNKS DEL CORPUS — chunk_fijo(texto, tamano=300, overlap=0)")
print("=" * 72)
for i, c in enumerate(chunks0):
    print(f"[{i}] ({origen0[i]}) {c[:70]}...")

rep0 = _reporte_idea(0)

# ----------------------------------------------------------------------
# La frase literal del enunciado y el corte exacto con overlap=0
# ----------------------------------------------------------------------
texto_vac = DOCS[IDEA_DOC]
f_ini = texto_vac.find(FRASE_CLAVE)
f_fin = f_ini + len(FRASE_CLAVE)
assert f_ini != -1, "No se encontró la frase clave"
filas_f80 = _clasificar_span(IDEA_DOC, f_ini, f_fin, OVERLAP)
filas_f0 = _clasificar_span(IDEA_DOC, f_ini, f_fin, 0)
frase_comp80 = [f["chunk"] for f in filas_f80 if f["estado"] == "completa"]
frase_cort80 = [f["chunk"] for f in filas_f80 if f["estado"] == "corta"]
frase_comp0 = [f["chunk"] for f in filas_f0 if f["estado"] == "completa"]
frase_cort0 = [f["chunk"] for f in filas_f0 if f["estado"] == "corta"]

izq = texto_vac[TAMANO - 35:TAMANO].replace("\n", " ")
der = texto_vac[TAMANO:TAMANO + 35].replace("\n", " ")

print("")
print("-" * 72)
print(f'Frase literal del enunciado: "{FRASE_CLAVE}" (chars {f_ini}-{f_fin - 1})')
print(f"  · overlap=80: COMPLETA en chunk {frase_comp80}; la corta: "
      f"{frase_cort80 if frase_cort80 else 'ninguno'}")
print(f"  · overlap=0 : COMPLETA en chunk {frase_comp0}; la corta: "
      f"{frase_cort0 if frase_cort0 else 'ninguno'}")
print(f"  · con overlap=0 el corte cae dentro de la oración: ...{izq} | {der}...")
print("    la frase sobrevive, pero la oración pierde su sujeto: la IDEA queda mutilada.")

# ----------------------------------------------------------------------
# Conclusión del ejercicio
# ----------------------------------------------------------------------
print("")
print("=" * 72)
print("CONCLUSIÓN (Ejercicio 1.1)")
print("=" * 72)
if rep80["cortan"]:
    print(f"· overlap=80: la idea SÍ es cortada por el chunk {rep80['cortan']}, "
          f"pero aparece COMPLETA en el chunk {rep80['completan']}: el "
          f"solapamiento de {OVERLAP} caracteres la salva.")
else:
    print("· overlap=80: ningún chunk corta la idea.")
if rep0["completan"]:
    print(f"· overlap=0: la idea aparece completa en el chunk {rep0['completan']}.")
else:
    print(f"· overlap=0: la idea queda mutilada: la cortan los chunks "
          f"{rep0['cortan']} y no aparece completa en ninguno (sin solapamiento, "
          "la idea que cruza la frontera se pierde a medias en cada fragmento).")

pequenos = [{"chunk": int(i), "origen": origen[i], "longitud": int(len(c)),
             "texto": c} for i, c in enumerate(chunks) if len(c) < 50]
if pequenos:
    det = "; ".join(f"[{p['chunk']}] ({p['origen']}) {p['longitud']} chars {p['texto']!r}"
                    for p in pequenos)
    print(f"· Artefacto del paso fijo de {TAMANO - OVERLAP} caracteres: chunk(s) "
          f"muy pequeño(s) al final -> {det}")

# ----------------------------------------------------------------------
# Cifras y resultados.json
# ----------------------------------------------------------------------
num_chunks = len(chunks)
num_chunks_0 = len(chunks0)
chunks_por_doc = {n: int(origen.count(n)) for n in DOCS}
chunks_por_doc_0 = {n: int(origen0.count(n)) for n in DOCS}
long_docs = {n: int(len(t)) for n, t in DOCS.items()}
long_chunks = [int(len(c)) for c in chunks]
long_chunks0 = [int(len(c)) for c in chunks0]
caracteres_corpus = int(sum(long_docs.values()))
caracteres_chunks80 = int(sum(long_chunks))
caracteres_chunks0 = int(sum(long_chunks0))
inflacion_teorica = OVERLAP / (TAMANO - OVERLAP)          # V/(C-V), slide 14
inflacion_medida80 = caracteres_chunks80 / caracteres_corpus - 1.0
inflacion_medida0 = caracteres_chunks0 / caracteres_corpus - 1.0

resultados = {
    # ---- cifras esperadas para esta subtarea ----
    "num_chunks": int(num_chunks),
    "chunks_por_documento": chunks_por_doc,
    "idea_cortada_con_overlap_80": bool(idea_cortada_80),
    "num_chunks_con_overlap_0": int(num_chunks_0),
    # ---- parámetros ----
    "parametros": {"tamano_caracteres": int(TAMANO),
                   "overlap_caracteres": int(OVERLAP)},
    # ---- cifras complementarias ----
    "longitud_documentos_caracteres": long_docs,
    "longitud_chunks_overlap80_caracteres": long_chunks,
    "longitud_chunks_overlap0_caracteres": long_chunks0,
    "chunks_por_documento_overlap_0": chunks_por_doc_0,
    "caracteres_corpus_total": caracteres_corpus,
    "caracteres_en_chunks_overlap80": caracteres_chunks80,
    "caracteres_en_chunks_overlap0": caracteres_chunks0,
    "inflacion_teorica_indice_overlap80_frac": float(inflacion_teorica),
    "inflacion_teorica_indice_overlap80_pct": float(inflacion_teorica * 100.0),
    "inflacion_medida_corpus_overlap80_frac": float(inflacion_medida80),
    "inflacion_medida_corpus_overlap0_frac": float(inflacion_medida0),
    "idea_doc": IDEA_DOC,
    "idea_texto": rep80["idea"],
    "idea_span_caracteres": [rep80["ini"], rep80["fin"]],
    "idea_cortada_con_overlap_0": bool(len(rep0["cortan"]) > 0),
    "idea_completa_en_algun_chunk_overlap_80": bool(len(rep80["completan"]) > 0),
    "idea_completa_en_algun_chunk_overlap_0": bool(len(rep0["completan"]) > 0),
    "idea_cortada_por_chunks_overlap_80": [int(i) for i in rep80["cortan"]],
    "idea_completa_en_chunks_overlap_80": [int(i) for i in rep80["completan"]],
    "idea_cortada_por_chunks_overlap_0": [int(i) for i in rep0["cortan"]],
    "idea_completa_en_chunks_overlap_0": [int(i) for i in rep0["completan"]],
    "frase_clave": FRASE_CLAVE,
    "frase_clave_span_caracteres": [int(f_ini), int(f_fin)],
    "frase_clave_completa_en_overlap_80": [int(i) for i in frase_comp80],
    "frase_clave_cortada_en_overlap_80": [int(i) for i in frase_cort80],
    "frase_clave_completa_en_overlap_0": [int(i) for i in frase_comp0],
    "frase_clave_cortada_en_overlap_0": [int(i) for i in frase_cort0],
    "chunks_menos_de_50_caracteres": pequenos,
    "preview_70chars_chunks_overlap80": [c[:70] for c in chunks],
    "preview_70chars_chunks_overlap0": [c[:70] for c in chunks0],
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print("")
print("=" * 72)
print("CIFRAS PRINCIPALES (guardadas en resultados.json)")
print("=" * 72)
print(f"num_chunks (overlap=80): {num_chunks}")
print(f"chunks_por_documento: {chunks_por_doc}")
print(f"idea_cortada_con_overlap_80: {idea_cortada_80}")
print(f"num_chunks_con_overlap_0: {num_chunks_0}")
print(f"  · longitudes de documentos (caracteres): {long_docs}")
print(f"  · caracteres: corpus={caracteres_corpus}, en chunks overlap=80: "
      f"{caracteres_chunks80}, en chunks overlap=0: {caracteres_chunks0}")
print(f"  · inflación teórica del índice C/(C-V): +{inflacion_teorica * 100:.4f} % "
      "(slide 14: +36.4 %)")
print(f"  · inflación medida en este corpus: overlap=80: "
      f"+{inflacion_medida80 * 100:.4f} % | overlap=0: "
      f"+{inflacion_medida0 * 100:.4f} %")
print("OK: resultados.json escrito en la carpeta actual.")

os.chdir(RAIZ)
mostrar_figuras(RAIZ / 'trabajo_nb' / 'T1')

CHUNKS DEL CORPUS — chunk_fijo(texto, tamano=300, overlap=80)
[0] (politica_vacaciones.md) Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo ...
[1] (politica_vacaciones.md) pación a través del portal interno. Los días no utilizados pueden
tran...
[2] (politica_vacaciones.md) meses....
[3] (politica_remoto.md) Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permi...
[4] (politica_remoto.md) ra trabajar desde el exterior del país se requiere aprobación
de Recur...
[5] (gastos.md) Política de reembolso de gastos de NimbusSoft.
Los gastos de viaje se ...
[6] (gastos.md)  deben comprarse en clase económica salvo vuelos de más de
8 horas, do...

------------------------------------------------------------------------
¿Algún chunk corta la idea? (overlap=80)
------------------------------------------------------------------------
Idea buscada en 'politica_vacaciones.md', caracteres 256 a 341:
  "Los días no utilizados pueden transferirse al año siguiente 

### Ejercicio 1.1 — `chunk_fijo(texto, tamano=300, overlap=80)`

Implementación sobre caracteres: la función recorre cada texto en ventanas de `tamano=300` caracteres con un solapamiento de `overlap=80` caracteres entre ventanas consecutivas y devuelve una lista de strings. El corpus fragmentado vive fuera del ejercicio porque las Partes 2 y 3 lo consumen: por cada documento se añaden sus chunks a `chunks` y su nombre a `origen`, y se imprime cada chunk con su índice, su documento de origen y una vista previa de 70 caracteres.

### Resultados medidos

Corpus: 3 documentos con **1165 caracteres** en total — `politica_vacaciones.md` (446), `politica_remoto.md` (361) y `gastos.md` (358).

| Métrica | overlap = 80 | overlap = 0 |
|---|---|---|
| Número de chunks | 7 | 6 |
| Chunks por documento | vacaciones: 3 · remoto: 2 · gastos: 2 | vacaciones: 2 · remoto: 2 · gastos: 2 |
| Longitudes de los chunks (caracteres) | 300, 226, 6, 300, 141, 300, 138 | 300, 146, 300, 61, 300, 58 |
| Caracteres dentro de los chunks | 1411 | 1165 |
| Inflación medida sobre el corpus | 0.2112 (21.1159 %) | 0.0 (0.0000 %) |

La **inflación teórica del índice** para overlap=80 es **0.3636** (36.3636 %; la diapositiva 14 del curso la redondea a +36.4 %): con solapamiento, cada chunk avanza C−V caracteres por paso, de modo que el índice puede crecer hasta C/(C−V). Lo medido fue menor, 0.2112, porque los últimos chunks de cada documento quedan parciales (226, 6, 141 y 138 caracteres) y duplican menos solape del que la fórmula asume.

### ¿Algún chunk corta una idea?

La idea crítica del corpus es: **«Los días no utilizados pueden transferirse al año siguiente hasta un máximo de 5 días.»** (en `politica_vacaciones.md`, span de caracteres [256, 342)).

- **Con overlap = 80:** la idea queda cortada por el chunk [0], pero aparece **COMPLETA en el chunk [1]**: el solapamiento de 80 caracteres la salva. La frase clave «hasta un máximo de 5 días» (span [316, 341)) también está completa en el chunk [1].
- **Con overlap = 0:** la idea queda mutilada: la cortan los chunks [0, 1] y **no aparece completa en ninguno** — sin solapamiento, la idea que cruza la frontera se pierde a medias en cada fragmento. La frase clave corta sí sobrevive entera en el chunk [1] (su span [316, 341) cae justo después de la frontera de 300), pero la idea completa que la rodea no.

Vista previa de los primeros chunks con overlap=80 (formato del enunciado, 70 caracteres):

- `[0] (politica_vacaciones.md) Política de vacaciones de NimbusSoft.` — arranca limpio al inicio del documento.
- `[1] (politica_vacaciones.md) pación a través del portal interno. Los días no utilizados pueden tran...` — empieza a mitad de una palabra («anticipación») y termina a mitad de la idea («transferirse»): es el solape trabajando.
- `[2] (politica_vacaciones.md) meses.` — el residuo del paso fijo.

### Artefacto del paso fijo

El paso efectivo es de **220 caracteres** (ventana de 300 − solape de 80). Con documentos de 358–446 caracteres, el residuo final puede quedar casi vacío: el chunk [2] (`politica_vacaciones.md`) mide solo **6 caracteres** — su texto completo es «meses.» —, por debajo del umbral de 50. Es el único chunk corto del índice (1 de 7), pero se embebe, se indexa, se recupera y se cita igual que cualquier otro sin contener idea alguna.

### Reflexión de la Parte 1

El fragmento es la unidad de todo el sistema: se embebe, se indexa con su identificador, se recupera y se cita en el prompt, y se compara con el golden set; cambiar el corte cambia los cuatro papeles a la vez (diapositiva 9). Y «ninguna forma de buscar arregla un fragmento mal cortado» (diapositiva 2): con overlap=0, la idea de los 5 días transferibles deja de existir como unidad y ningún kNN la devolvería completa. El overlap=80 la salva a un costo medido de 0.2112 de caracteres extra en el índice.

## Parte 2 — Índice (embeddings + kNN)

### ⚠️ Estado: NO EJECUTADA — presupuesto agotado

**Declaración obligatoria:** la subtarea T2 (índice de embeddings + kNN, techo del modelo y `recuperar()`, Ejercicio 2.1) **no pudo ejecutarse por presupuesto agotado**. En consecuencia **no hay ninguna cifra medida de esta parte**: se desconocen el motor efectivo (denso `sentence-transformers` o fallback léxico TF-IDF), la forma del índice `V.shape`, el tope `max_seq_length` del modelo, el número de tokens del fragmento más largo y los scores de similitud de cualquier consulta. Todas esas cifras faltan y así se declara; no se estiman ni se sustituyen.

### Lo que esta celda debía medir (según el enunciado)

- **Motor de retrieval:** `SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")` con embeddings normalizados si está instalado; si no, fallback `TfidfVectorizer` con vectores normalizados. La celda imprime `Retrieval: <motor> | índice: V.shape`.
- **El techo del modelo, medido:** el chunking de la Parte 1 cuenta caracteres y el modelo de embeddings trunca en tokens — el MiniLM multilingüe procesa como mucho `max_seq_length` tokens de cada texto y descarta el resto sin avisar. La celda cuenta los tokens del fragmento más largo con el tokenizador del mismo modelo y los compara con su tope, imprimiendo si «cabe entero» o cuántos tokens «se pierden en silencio». Como advierte el enunciado: trescientos caracteres probablemente caben, «pero "probablemente" no es una medición». (El Lab 02 del sábado evita esto fragmentando directo en tokens de bge-m3, servido en la H200 de la USFQ, con un tope de 8192.)
- Con el fallback TF-IDF no hay modelo que trunque: el fallback lee el fragmento entero.

### Ejercicio 2.1 — `recuperar(pregunta, k=3)` (pendiente)

Contrato pedido: `recuperar(pregunta, k=3) → lista de (score, idx_chunk)` ordenada de mayor a menor score, usando el índice `V` y `vec_consulta(pregunta)`. La Parte 3 y el Ejercicio 3.1 la llaman con esa firma exacta. Al no ejecutarse T2, esta función queda sin ejecutar y sin verificación, y la Parte 3 no puede correr: es el segundo efecto en cascada del presupuesto agotado.

## Parte 3 — Generación con contexto

### ⚠️ Estado: NO EJECUTADA — presupuesto agotado

**Declaración obligatoria:** la subtarea T3 (generación con contexto y las tres pruebas de fuego, Ejercicio 3.1) **no pudo ejecutarse por presupuesto agotado**. No hay respuestas medidas, no hay chunks recuperados anotados y no hay resultados del detector de abstención para ninguna de las tres preguntas. Las cifras de las tres pruebas de fuego faltan y se declaran ausentes; no se reemplazan por simulaciones.

### El diseño que la Parte 3 pone a prueba (según el enunciado)

- **Prompt del baseline:** contexto delimitado con `<contexto>...</contexto>`, instrucción de responder SOLO con base en él y permiso explícito de decir «no sé». `armar_prompt(pregunta, k=3)` inserta los k chunks que devuelve `recuperar`, cada uno citado como `[i]`.
- **Frase de abstención única en todo el curso:** `ABSTENCION = "El corpus no contiene información suficiente."` — la misma del Lab 02 (`rag_pipeline.py`) y del golden set del sábado; con tres redacciones distintas en tres archivos, ningún detector por igualdad de cadena dispararía nunca.
- **Detección de abstención normalizada:** `se_abstuvo()` compara normalizado (minúsculas, sin tildes, sin puntuación), de modo que «informacion suficiente» y «información suficiente.» cuentan como la misma frase.
- **Cadena de generación con fallback:** API de Anthropic (claude-haiku-4-5) → H200 de la USFQ (vLLM, API compatible con OpenAI, requiere GlobalProtect) → Ollama local (llama3.1:8b) → **modo inspección** con respuestas pre-capturadas (`transferir`, `exterior`, `mascotas`, `ambigua`). En modo inspección el sistema muestra el prompt armado y una respuesta pre-capturada para depurar el retrieval igual.

### Ejercicio 3.1 — las tres pruebas de fuego (pendiente)

1. Respuesta clara: «¿puedo trabajar desde el exterior?» (etiqueta `exterior`).
2. Sin respuesta en el corpus: «¿puedo llevar a mi mascota a la oficina?» (etiqueta `mascotas`).
3. Ambigua entre documentos: «¿qué necesita aprobación del gerente?» (etiqueta `ambigua`).

Para cada una el enunciado pide anotar: ¿qué chunks recuperó? ¿la respuesta se apoya en ellos? **Ninguna de las dos anotaciones existe: la ejecución quedó pendiente.**

Punto clave de la prueba 2 (citado del enunciado): el mejor comportamiento posible es la frase de abstención, `ABSTENCION` («El corpus no contiene información suficiente»). Un RAG que siempre responde algo es un generador de alucinaciones con citas decorativas.

## Diagnóstico final

**Pregunta del enunciado:** de las 4 cajas del pipeline (ingesta / chunking / retrieval / generación), ¿cuál te parece más frágil en TU proyecto y qué harías para vigilarla?

**Respuesta, con la única caja que hoy tiene cifras medidas:** el **chunking**. No es una intuición: es la única caja ejecutada, y ya con este corpus diminuto de 1165 caracteres mostró dos fallos medidos.

1. **Corta ideas en la frontera.** La idea «Los días no utilizados pueden transferirse al año siguiente hasta un máximo de 5 días» (span [256, 342) de `politica_vacaciones.md`) queda cortada por el chunk [0] y solo aparece completa en el chunk [1] gracias al solape de 80 caracteres; con overlap=0 la cortan los chunks [0, 1] y no aparece completa en ninguno. Como el fragmento es la unidad de todo el sistema —se embebe, se indexa, se recupera y se cita, y se compara con el golden set— y «ninguna forma de buscar arregla un fragmento mal cortado» (diapositivas 2 y 9), ese corte degradaría en silencio a las otras tres cajas.
2. **Produce residuos sin idea.** El paso fijo de 220 caracteres (ventana 300 − solape 80) dejó un chunk de 6 caracteres —«meses.», el chunk [2]—, por debajo del umbral de 50: ruido con identificador propio, recuperable y citable igual que un chunk bueno.

**Qué haría para vigilarla:**

- Medir tras cada cambio de parámetros, no suponer: hoy el índice pasó de 1165 a 1411 caracteres (inflación medida 0.2112 con overlap=80; teórica 0.3636) y de 6 a 7 chunks. Esos números son el precio defendible del solape.
- Añadir un chequeo automático de sanidad: ningún chunk por debajo de 50 caracteres (hoy falla: 1 de 7) y cada idea clave del corpus completa en algún chunk (con overlap=80 cumple; con overlap=0 falla).
- Justificar los parámetros contra algo medido — «fragmentar con parámetros que sepas defender» (diapositiva 28) — y dejar que la evaluación de mañana (Hit Rate, MRR, Recall@k sobre un golden set) arbitre.

**Límite honesto de esta conclusión:** no puedo comparar la fragilidad del chunking con la del retrieval ni con la de la generación, porque T2 y T3 no se ejecutaron (presupuesto agotado). La conclusión se limita a lo medido: de las cajas con cifras, el chunking es la más frágil — y es, además, la que acota a las demás.

## Cierre

Lo construido y medido: un chunking fijo por caracteres con solapamiento, evaluado con y sin overlap sobre el corpus de 3 documentos (1165 caracteres). Cifras finales de la Parte 1: **7 chunks con overlap=80** (longitudes 300, 226, 6, 300, 141, 300, 138; 1411 caracteres; inflación 0.2112) frente a **6 chunks con overlap=0** (longitudes 300, 146, 300, 61, 300, 58; 1165 caracteres; inflación 0.0). El solape salva la idea de los 5 días transferibles —completa en el chunk [1]— a costa de 0.2112 de caracteres extra indexados; sin solape, esa idea no aparece completa en ningún chunk.

Lo que falta (declarado): la Parte 2 —índice, techo del modelo y `recuperar()`— y la Parte 3 —generación con contexto y las tres pruebas de fuego— no se ejecutaron por **presupuesto agotado**. El pipeline queda armado y medido hasta la caja de chunking; índice, retrieval y generación quedan sin cifras.

Mañana (enunciado, S4): ¿este RAG es bueno? Métricas de retrieval (Hit Rate, MRR, Recall@k), golden sets, y las extensiones que compiten contra tu baseline. Y el Taller 2 (25 %, sábado 26 de septiembre) pide «fragmentar con parámetros explícitos y justificados»: hoy vimos contra qué se justifican — las cifras de arriba.